# Imports

In [44]:
import re
import torch
import optuna
import pickle

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn import set_config
from catboost import CatBoostClassifier
from category_encoders import CatBoostEncoder

from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.preprocessing import TargetEncoder, StandardScaler, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict

from feature_engine.selection import DropFeatures
from pytorch_tabnet.tab_model import TabNetClassifier

from tensorflow.keras import layers
from tensorflow.keras import callbacks
from tensorflow.keras import regularizers
from tensorflow.keras.models import Sequential

I0000 00:00:1778800699.793923  184631 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Utils

In [31]:
set_config(enable_metadata_routing=True)

In [32]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

In [33]:
column_transformer = ColumnTransformer([
    (
        'target_encoder', 
        TargetEncoder(), 
        ['driver', 'compound', 'race']
    ),
    (
        'catboost_encoder', 
        CatBoostEncoder(), 
        ['driver', 'compound', 'race']
    ),
    (
        'standard_scaler', 
        StandardScaler(), 
        ['lapnumber', 'position', 'raceprogress', 'year', 'position_norm', 'race_progress_sin', 'position_vs_mean']
    ),
    (
        'robust_scaler', 
        RobustScaler(), 
        [
            'position_change', 'cumulative_degradation', 'laptime_delta', 'laptime_s', 'stint', 'driver_mean_lap', 'tyrelife', 'delta_x_tyre_life', 
            'compound_tyre_life', 'stint_progress', 'tyre_life_ratio', 'degradation_per_lap', 'position_change_cum', 'laps_since_pit', 'lap_time_inv',  
            'lap_time_vs_race_mean', 'lap_time_x_tyre', 'position_x_progress', 'degradation_x_progress', 'race_progress_squared', 'driver_avg_position' 
        ]
    ),
], remainder="passthrough")

# Loading Dataset

In [34]:
X_train = pd.read_parquet('../data/X_train.parquet')
X_test = pd.read_parquet('../data/X_test.parquet')

y_train = pd.read_parquet('../data/y_train.parquet')

In [35]:
X_train.head()

,driver,compound,race,year,pitstop,lapnumber,stint,tyrelife,position,laptime_s,...,treeraceprogress,treeposition,treelaptime_s,treecumulative_degradation,treeraceprogress_position,treeraceprogress_laptime_s,treeraceprogress_cumulative_degradation,treeposition_laptime_s,treeposition_cumulative_degradation,treelaptime_s_cumulative_degradation
id,,,,,,,,,,,,,,,,,,,,,
0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,...,0.298107,0.207392,0.252153,0.279021,0.298107,0.298107,0.321361,0.252153,0.279021,0.253721
1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,...,0.298107,0.185508,0.252153,0.586119,0.298107,0.298107,0.610176,0.252153,0.544023,0.613159
2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,...,0.298107,0.207392,0.252153,0.207676,0.298107,0.298107,0.423519,0.252153,0.228194,0.231339
3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,...,0.102471,0.185508,0.178575,0.138352,0.102471,0.102471,0.043550,0.178575,0.138352,0.070991
4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,...,0.298107,0.185508,0.178575,0.138352,0.298107,0.298107,0.206543,0.178575,0.138352,0.153276


In [36]:
X_test.head()

,driver,compound,race,year,pitstop,lapnumber,stint,tyrelife,position,laptime_s,...,treeraceprogress,treeposition,treelaptime_s,treecumulative_degradation,treeraceprogress_position,treeraceprogress_laptime_s,treeraceprogress_cumulative_degradation,treeposition_laptime_s,treeposition_cumulative_degradation,treelaptime_s_cumulative_degradation
id,,,,,,,,,,,,,,,,,,,,,
439140,D119,MEDIUM,British Grand Prix,2023,0,21,1,21.0,4,93.387,...,0.298107,0.185508,0.178575,0.138352,0.298107,0.298107,0.206543,0.178575,0.138352,0.070991
439141,VER,MEDIUM,Abu Dhabi Grand Prix,2023,0,24,1,24.0,1,90.867,...,0.298107,0.185508,0.178575,0.147360,0.298107,0.298107,0.321361,0.178575,0.147360,0.253721
439142,D270,MEDIUM,British Grand Prix,2023,0,24,1,24.0,11,92.871,...,0.298107,0.207392,0.178575,0.138352,0.298107,0.298107,0.206543,0.178575,0.138352,0.070991
439143,D112,SOFT,São Paulo Grand Prix,2024,0,6,2,4.0,15,94.967,...,0.102471,0.207392,0.178575,0.279021,0.102471,0.102471,0.071042,0.178575,0.279021,0.216120
439144,AND,HARD,United States Grand Prix,2024,0,52,2,29.0,12,99.112,...,0.298107,0.207392,0.178575,0.111582,0.298107,0.298107,0.206543,0.178575,0.111582,0.216120


## Creating Stacking Database

In [7]:
lgbm = load_pickle('../models/model_lightgbm.pkl')
cat = load_pickle('../models/model_pure_catboost.pkl')
catencoder_cat = load_pickle('../models/model_catboostencoder_catboost.pkl')
xgb = load_pickle('../models/model_xgboost.pkl')
hist = load_pickle('../models/model_histgradientboosting.pkl')
lg = load_pickle('../models/model_logistic_regression.pkl')
voting = load_pickle('../models/model_voting.pkl')

In [9]:
cv = StratifiedKFold(shuffle=True, random_state=42, n_splits=5)

lgbm_proba = cross_val_predict(lgbm, X_train, y_train.PitNextLap, cv=cv, n_jobs=-1, method='predict_proba')[:, 1]
catencoder_cat_proba = cross_val_predict(catencoder_cat, X_train, y_train.PitNextLap, cv=cv, n_jobs=-1, method='predict_proba')[:, 1]
xgb_proba = cross_val_predict(xgb, X_train, y_train.PitNextLap, cv=cv, n_jobs=-1, method='predict_proba')[:, 1]
hist_proba = cross_val_predict(hist, X_train, y_train.PitNextLap, cv=cv, n_jobs=-1, method='predict_proba')[:, 1]
lg_proba = cross_val_predict(lg, X_train, y_train.PitNextLap, cv=cv, n_jobs=-1, method='predict_proba')[:, 1]
voting_proba = cross_val_predict(voting, X_train, y_train.PitNextLap, cv=cv, n_jobs=-1, method='predict_proba',)[:, 1]

In [39]:
oof_preds = np.zeros(len(X_train))
test_preds = np.zeros(len(X_test))

for fold, (train_idx, valid_idx) in enumerate(cv.split(X_train, y_train)):

    print(f"\n======== Fold {fold + 1} ========")

    X_tr = X_train.iloc[train_idx, :]
    X_val = X_train.iloc[valid_idx, :]

    y_tr = y_train.iloc[train_idx, 0]
    y_val = y_train.iloc[valid_idx, 0]

    X_tr_transformed = column_transformer.fit_transform(X_tr, y_tr)
    X_val_transformed = column_transformer.transform(X_val)
    X_test_transformed = column_transformer.transform(X_test)

    tab_net = TabNetClassifier(
        n_d=32,
        n_a=32,
        n_steps=5,
        gamma=1.5,
        lambda_sparse=1e-4,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=1e-2),
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        scheduler_params={"step_size": 20, "gamma": 0.9},
        mask_type="entmax",
        seed=42,
        verbose=1
    )

    tab_net.fit(
        X_train=X_tr_transformed,
        y_train=y_tr.values,
        eval_set=[(X_val_transformed, y_val.values)],
        eval_name=["valid"],
        eval_metric=["auc"],
        max_epochs=200,
        patience=20,
        batch_size=1024,
        virtual_batch_size=128,
        num_workers=11,
        drop_last=False
    )

    oof_preds[valid_idx] = tab_net.predict_proba(X_val_transformed)[:, 1]

fold_test_preds = tab_net.predict_proba(X_test_transformed)[:, 1]


======== Fold 1 ========


/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.38938 | valid_auc: 0.87697 |  0:00:18s
epoch 1  | loss: 0.31181 | valid_auc: 0.90711 |  0:00:36s
epoch 2  | loss: 0.29353 | valid_auc: 0.91508 |  0:00:55s
epoch 3  | loss: 0.28634 | valid_auc: 0.91695 |  0:01:13s
epoch 4  | loss: 0.28057 | valid_auc: 0.92062 |  0:01:31s
epoch 5  | loss: 0.27775 | valid_auc: 0.92153 |  0:01:50s
epoch 6  | loss: 0.27262 | valid_auc: 0.92349 |  0:02:09s
epoch 7  | loss: 0.27408 | valid_auc: 0.92603 |  0:02:27s
epoch 8  | loss: 0.27558 | valid_auc: 0.91391 |  0:02:46s
epoch 9  | loss: 0.27669 | valid_auc: 0.92439 |  0:03:05s
epoch 10 | loss: 0.27044 | valid_auc: 0.9255  |  0:03:25s
epoch 11 | loss: 0.2665  | valid_auc: 0.92826 |  0:03:44s
epoch 12 | loss: 0.26389 | valid_auc: 0.9234  |  0:04:03s
epoch 13 | loss: 0.26803 | valid_auc: 0.92958 |  0:04:22s
epoch 14 | loss: 0.26174 | valid_auc: 0.92913 |  0:04:41s
epoch 15 | loss: 0.26173 | valid_auc: 0.93231 |  0:05:00s
epoch 16 | loss: 0.25639 | valid_auc: 0.9342  |  0:05:19s
epoch 17 | los

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



======== Fold 2 ========


...

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



======== Fold 5 ========


/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.38905 | valid_auc: 0.87773 |  0:00:19s
epoch 1  | loss: 0.31637 | valid_auc: 0.90294 |  0:00:38s
epoch 2  | loss: 0.30915 | valid_auc: 0.90165 |  0:00:57s
epoch 3  | loss: 0.29954 | valid_auc: 0.91467 |  0:01:17s
epoch 4  | loss: 0.28454 | valid_auc: 0.91934 |  0:01:36s
epoch 5  | loss: 0.27706 | valid_auc: 0.92302 |  0:01:56s
epoch 6  | loss: 0.27446 | valid_auc: 0.92351 |  0:02:14s
epoch 7  | loss: 0.27053 | valid_auc: 0.92437 |  0:02:32s
epoch 8  | loss: 0.26812 | valid_auc: 0.92652 |  0:02:51s
epoch 9  | loss: 0.26637 | valid_auc: 0.9272  |  0:03:09s
epoch 10 | loss: 0.26574 | valid_auc: 0.93086 |  0:03:27s
epoch 11 | loss: 0.26164 | valid_auc: 0.9292  |  0:03:46s
epoch 12 | loss: 0.26071 | valid_auc: 0.93105 |  0:04:04s
epoch 13 | loss: 0.26344 | valid_auc: 0.93129 |  0:04:23s
epoch 14 | loss: 0.25836 | valid_auc: 0.9305  |  0:04:42s
epoch 15 | loss: 0.25614 | valid_auc: 0.93315 |  0:05:01s
epoch 16 | loss: 0.25292 | valid_auc: 0.93519 |  0:05:20s
epoch 17 | los

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [48]:
nn_oof_train = np.zeros(len(X_train))
nn_test_preds = np.zeros(len(X_test))

aucs = []

cv = StratifiedKFold(shuffle=True, random_state=42, n_splits=3)

for fold, (train_idx, valid_idx) in enumerate(cv.split(X_train, y_train)):
    print(f"\n======== Fold {fold + 1} ========")

    X_tr, X_val = X_train.iloc[train_idx, :], X_train.iloc[valid_idx, :]
    y_tr, y_val = y_train.iloc[train_idx, 0], y_train.iloc[valid_idx, 0]

    X_tr_tf = column_transformer.fit_transform(X_tr, y_tr).astype(np.float32)
    X_val_tf = column_transformer.transform(X_val).astype(np.float32)
    X_test_tf = column_transformer.transform(X_test).astype(np.float32)

    y_tr = y_tr.values.astype(np.float32)
    y_val = y_val.values.astype(np.float32)

    model = Sequential([
        layers.Input(shape=(X_tr_tf.shape[1],)),
        layers.Dense(512, activation="swish", kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Dropout(0.30),
        layers.Dense(256, activation="swish", kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Dropout(0.25),
        layers.Dense(128, activation="swish", kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Dropout(0.20),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(name="auc")]
    )

    early_stopping = callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=15, restore_best_weights=True)
    reduce_lr = callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.5, patience=5, min_lr=1e-6)

    model.fit(
        X_tr_tf, y_tr,
        validation_data=(X_val_tf, y_val),
        epochs=200,
        batch_size=1024,
        callbacks=[early_stopping, reduce_lr],
        verbose=1
    )
    
    val_preds = model.predict(X_val_tf, verbose=0).ravel()
    nn_oof_train[valid_idx] = val_preds
    
    nn_test_preds += model.predict(X_test_tf, verbose=0).ravel() / cv.get_n_splits()

    auc = roc_auc_score(y_val, val_preds)
    aucs.append(auc)
    print(f"Fold AUC: {auc:.6f}")

print(f"\nMean AUC: {np.mean(aucs):.6f} (+/- {np.std(aucs):.6f})")


======== Fold 1 ========
Epoch 1/200
286/286 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - auc: 0.8563 - loss: 0.4502 - val_auc: 0.9075 - val_loss: 0.3819 - learning_rate: 0.0010
Epoch 2/200
286/286 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - auc: 0.9099 - loss: 0.3456 - val_auc: 0.9186 - val_loss: 0.3367 - learning_rate: 0.0010
Epoch 3/200
286/286 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - auc: 0.9194 - loss: 0.3215 - val_auc: 0.9248 - val_loss: 0.3161 - learning_rate: 0.0010
Epoch 4/200
286/286 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - auc: 0.9231 - loss: 0.3081 - val_auc: 0.9262 - val_loss: 0.3032 - learning_rate: 0.0010
Epoch 5/200
286/286 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - auc: 0.9254 - loss: 0.2982 - val_auc: 0.9288 - val_loss: 0.2940 - learning_rate: 0.0010
Epoch 6/200
286/286 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - auc: 0.9267 - loss: 0.2914 - val_auc: 0.9301 - val_loss: 0.2856 - learning_rate: 0.0010
Epoch 7/200
286/286 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - auc: 0.9273 - loss: 0.2869 - val_auc: 0.9313 - val_los

In [49]:
X_train_stacking = pd.DataFrame({
    'lgbm_proba': lgbm_proba,
    'catencoder_cat_proba': catencoder_cat_proba,
    'xgb_proba': xgb_proba,
    'hist_proba': hist_proba,
    'lg_proba': lg_proba,
    'voting_proba': voting_proba,
    'cat_proba': cat_proba,
    'tabnet_proba': oof_preds,
    'nn_proba': nn_oof_train
})

X_train_stacking.to_parquet('../data/X_train_4.1_stacking.parquet')

In [50]:
X_test_stacking = pd.DataFrame({
    'lgbm_proba': lgbm.predict_proba(X_test)[:, 1],
    'catencoder_cat_proba': catencoder_cat.predict_proba(X_test)[:, 1],
    'xgb_proba': xgb.predict_proba(X_test)[:, 1],
    'hist_proba': hist.predict_proba(X_test)[:, 1],
    'lg_proba': lg.predict_proba(X_test)[:, 1],
    'voting_proba': voting.predict_proba(X_test)[:, 1],
    'cat_proba': cat.predict_proba(X_test)[:, 1],
    'tabnet_proba': test_preds,
    'nn_proba': nn_test_preds
})

X_test_stacking.to_parquet('../data/X_test_4.1_stacking.parquet')

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


# Machine Learning

In [52]:
def objective(trial, X, y):

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    aucs = []

    params = {
        "solver": trial.suggest_categorical("solver", ["saga"]),
        "C": trial.suggest_float("C", 1e-5, 100, log=True),
        "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0),
        "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"]),
        "fit_intercept": trial.suggest_categorical("fit_intercept", [True, False]),
        "tol": trial.suggest_float("tol", 1e-6, 1e-2, log=True),
        "max_iter": trial.suggest_int("max_iter", 1000, 5000),
        "random_state": trial.suggest_int("random_state", 0, 1000)
    }

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):

        X_train, X_valid = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train, y_valid = y.iloc[train_idx, 0], y.iloc[valid_idx, 0]

        model = LogisticRegression(**params)
        model.fit(X_train, y_train)

        proba = model.predict_proba(X_valid)[:, 1]

        auc = roc_auc_score(y_valid, proba)
        aucs.append(auc)

        trial.report(np.mean(aucs), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(aucs)


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(lambda trial: objective(trial, X_train_stacking, y_train), n_trials=500, n_jobs=-1)


print("Best trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-05-14 21:28:55,483] A new study created in memory with name: no-name-d9cc86ca-fc4f-47a0-b0d2-e1560ae76168
[I 2026-05-14 21:29:04,675] Trial 1 finished with value: 0.9024483204954754 and parameters: {'solver': 'saga', 'C': 0.005941338610121171, 'l1_ratio': 0.40209120383159525, 'class_weight': 'balanced', 'fit_intercept': False, 'tol': 0.009705910923974045, 'max_iter': 4609, 'random_state': 6}. Best is trial 1 with value: 0.9024483204954754.
[I 2026-05-14 21:29:10,224] Trial 2 finished with value: 0.9491305441681325 and parameters: {'solver': 'saga', 'C': 6.057123968355615e-05, 'l1_ratio': 0.09860942745730295, 'class_weight': 'balanced', 'fit_intercept': False, 'tol': 0.000973816156801536, 'max_iter': 2339, 'random_state': 736}. Best is trial 2 with value: 0.9491305441681325.
[I 2026-05-14 21:29:12,107] Trial 9 finished with value: 0.9495357436760814 and parameters: {'solver': 'saga', 'C': 18.620516949192602, 'l1_ratio': 0.19156575734480408, 'class_weight': 'balanced', 'fit_inter

Best trial score:
0.9497659710345386

Best params:
{'solver': 'saga', 'C': 0.008630817906865327, 'l1_ratio': 0.8250356179011763, 'class_weight': None, 'fit_intercept': True, 'tol': 9.210749213449445e-06, 'max_iter': 4329, 'random_state': 736}


In [53]:
stacking_lg = LogisticRegression(**study.best_params).fit(X_train_stacking, y_train.PitNextLap)

# Submission

In [54]:
submission = pd.read_csv('../data/sample_submission.csv')

In [55]:
submission['PitNextLap'] = stacking_lg.predict_proba(X_test_stacking)[:, 1]

In [56]:
submission.to_csv('../data/submission.csv', index=False)